In [1]:
import pandas as pd

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

In [2]:
from pathlib import Path

google_api_key_file = "google_api.key" # file containing google API key for geocoding
facility_file_url = "https://www.dshs.texas.gov/sites/default/files/thcic/hospitals/facilitylist.xlsx"
provider_output_file = "Facility_type1q2016_tab_ZIP.csv" # output file containing THCIC_ID, PROVIDER_NAME and ZIP code

# Check for the existence of the output file. If it exists, read from it; otherwise, read from the input file and process to get ZIP codes.
file_path = Path(provider_output_file)

if not file_path.is_file():
    provider_input_file = "Facility_type1q2016_tab.txt" # file containing THCIC_ID and PROVIDER_NAME from PUDF data, may not be the same as facility list in the excel file above
    provider_input_file_sep = '\t'
else:
    provider_input_file = provider_output_file
    provider_input_file_sep = ','

# --- Set API key ---
with open(google_api_key_file, "r") as file:
    API_KEY = file.read()


We download the facility list file and use it to associate the provider with its geographical (zip code, city and county)

In [3]:

# excel file containing facility name and zip code and address and thcic_id, may not be the same as provider list in PUDF data
# df = pd.read_excel("https://www.dshs.texas.gov/sites/default/files/thcic/hospitals/facilitylist.xlsx")
df = pd.read_excel(facility_file_url)
df_facility = df[["THCIC ID","Facility","City","County","ZIP"]]
df_facility.columns = ["THCIC_ID","PROVIDER_NAME","CITY","COUNTY","ZIP"]
del df

df = pd.read_csv(provider_input_file,sep=provider_input_file_sep) # file containing THCIC_ID and PROVIDER_NAME from PUDF data
if "ZIP" not in df.columns:
    df_provider = df[["THCIC_ID","PROVIDER_NAME"]]
else:
    df_provider = df[["THCIC_ID","PROVIDER_NAME","ZIP"]]
    df_provider["ZIP_OLD"] = df_provider["ZIP"] 
    df_provider["ZIP"] = df_provider["ZIP"].astype(str)
    df_provider["ZIP_OLD"] = df_provider["ZIP_OLD"].astype(str)
del df

df_provider.head()

,THCIC_ID,PROVIDER_NAME,ZIP,ZIP_OLD
0,100,Austin State Hospital,78751,78751
1,101,Big Spring State Hospital,79702,79702
2,102,UT Medical Branch Hospital,77550,77550
3,104,Rio Grande State Center,78552,78552
4,105,UT MD Anderson Cancer Center,77030,77030


Step 1: We map the providers to their zip code, city and county using the facility data from TX DSHS. Some entries may not be mapped. 

In [4]:
# features = ["ZIP","CITY","COUNTY"]
features = ["ZIP"]    

for addr_feature in features:
    # if addr_feature not in df_provider.columns:    
    #     df_provider[addr_feature] = "UNKNOWN"
    facility_dict = df_facility.set_index("PROVIDER_NAME")[addr_feature].to_dict()
    df_provider[addr_feature] = df_provider["PROVIDER_NAME"].map(facility_dict)
    df_provider.loc[df_provider[addr_feature].isna(), addr_feature] = "UNKNOWN"  

if "ZIP_OLD" in df_provider.columns:
    if "UNKNOWN" in df_provider["ZIP"].values:
        df_provider.loc[df_provider["ZIP"]=="UNKNOWN","ZIP"] = df_provider.loc[df_provider["ZIP"]=="UNKNOWN","ZIP_OLD"]
    df_provider.drop(columns=["ZIP_OLD"], inplace=True)

if "UNKNOWN" in df_provider["ZIP"].values:
    num_unknown_zip = df_provider["ZIP"].value_counts()["UNKNOWN"]
else:
    num_unknown_zip = 0    
print(f"Number of providers with known ZIP codes: {len(df_provider) - num_unknown_zip}")
print(f"Number of providers with unknown ZIP codes: {num_unknown_zip}")

Number of providers with known ZIP codes: 697
Number of providers with unknown ZIP codes: 1


Step 1a: we use the handcoded data from the previous quarters to fill in the zip info


In [ ]:
from pathlib import Path

facility_filenames = ['provider_zip_codes/Facility_type1q2019_tab_ZIP_handcoded.csv',\
                      'provider_zip_codes/Facility_type2q2019_tab_ZIP_handcoded.csv',\
                      'provider_zip_codes/Facility_type3q2019_tab_ZIP_handcoded.csv',\
                      'provider_zip_codes/Facility_type4q2019_tab_ZIP_handcoded.csv',\
                      'provider_zip_codes/Facility_type1q2018_tab_ZIP_handcoded.csv',\
                      'provider_zip_codes/Facility_type2q2018_tab_ZIP_handcoded.csv',\
                      'provider_zip_codes/Facility_type3q2018_tab_ZIP_handcoded.csv',\
                      'provider_zip_codes/Facility_type4q2018_tab_ZIP_handcoded.csv',\
                      'provider_zip_codes/Facility_type1q2017_tab_ZIP_handcoded.csv',\
                      'provider_zip_codes/Facility_type2q2017_tab_ZIP_handcoded.csv',\
                      'provider_zip_codes/Facility_type3q2017_tab_ZIP_handcoded.csv',\
                      'provider_zip_codes/Facility_type4q2017_tab_ZIP_handcoded.csv',\
                      'provider_zip_codes/Facility_type1q2016_tab_ZIP_handcoded.csv',\
                      'provider_zip_codes/Facility_type2q2016_tab_ZIP_handcoded.csv',\
                      'provider_zip_codes/Facility_type3q2016_tab_ZIP_handcoded.csv',\
                      'provider_zip_codes/Facility_type4q2016_tab_ZIP_handcoded.csv']


for filename in facility_filenames:
    file_path = Path(filename)
    if file_path.is_file():
        df_temp = pd.read_csv(filename)
        df_temp["ZIP"] = df_temp["ZIP"].astype(str)
        facility_dict = df_temp.set_index("PROVIDER_NAME")["ZIP"].to_dict()
        bit_mask = df_provider["ZIP"]=="UNKNOWN"
        df_provider.loc[bit_mask,"ZIP"] = df_provider.loc[bit_mask,"PROVIDER_NAME"].map(facility_dict)
        df_provider.loc[df_provider["ZIP"].isna(), "ZIP"] = "UNKNOWN"  

if "UNKNOWN" in df_provider["ZIP"].values:
    num_unknown_zip = df_provider["ZIP"].value_counts()["UNKNOWN"]
else:
    num_unknown_zip = 0    
print(f"Number of providers with known ZIP codes: {len(df_provider) - num_unknown_zip}")
print(f"Number of providers with unknown ZIP codes: {num_unknown_zip}")

Number of providers with known ZIP codes: 698
Number of providers with unknown ZIP codes: 0


Step 2: We use the Geocoding API from Google to find the zip codes for the remaining providers that did not get mapped to an address

In [6]:
import requests
import time

def get_hospital_address_and_zip(hospital_name, api_key):
    # Target URL for Geocoding API
    base_url = "https://maps.googleapis.com/maps/api/geocode/json"
    
    # Combine hospital name and city to make the search highly accurate
    search_query = f"{hospital_name}, TX"
    
    params = {
        "address": search_query,
        "key": api_key
    }
    
    try:
        response = requests.get(base_url, params=params)
        data = response.json()
        
        if data["status"] == "OK":
            # Extract the first match found by Google
            result = data["results"][0]
            formatted_address = result["formatted_address"]
            
            # Loop through address components to find the 'postal_code'
            zip_code = "UNKNOWN"
            for component in result["address_components"]:
                if "postal_code" in component["types"]:
                    zip_code = component["long_name"]
                    break
                    
            return zip_code, formatted_address
            
        elif data["status"] == "ZERO_RESULTS":
            return "UNKNOWN", "UNKNOWN"
        else:
            print(f"API Error Status: {data['status']}")
            return "UNKNOWN", "UNKNOWN"
            
    except Exception as e:
        print(f"Request failed: {e}")
        return "UNKNOWN", "UNKNOWN"


In [7]:

for idx, provider in enumerate(df_provider.loc[0:,"PROVIDER_NAME"]):    
    if df_provider.loc[idx,"ZIP"] == "UNKNOWN":
        zip_code, full_address = get_hospital_address_and_zip(provider, API_KEY)
        print(idx)
        print(f"Hospital: {provider}")
        print(f"Zip Code: {zip_code}")
        print(f"Full Address: {full_address}")
        print("-" * 40)    
        df_provider.loc[idx,"ZIP"] = zip_code
        time.sleep(0.1)
    # Pause slightly between requests to respect basic rate limit


In [8]:
# df_provider['ZIP'].value_counts()
df_provider.head(20)

,THCIC_ID,PROVIDER_NAME,ZIP
0,100,Austin State Hospital,78751
1,101,Big Spring State Hospital,79702
2,102,UT Medical Branch Hospital,77550
3,104,Rio Grande State Center,78552
4,105,UT MD Anderson Cancer Center,77030
5,106,Kerrville State Hospital,78028
6,107,Rusk State Hospital,75785
7,108,Texas Center for Infectious Disease,78223
8,110,San Antonio State Hospital,78223
9,111,Terrell State Hospital,75160


Check how many missing data points we have. The UNKNOWN zip codes will have to be inputed the old-fashioned way, by hand unfortunately

In [9]:
if "UNKNOWN" in df_provider["ZIP"].values:
    num_unknown_zip = df_provider["ZIP"].value_counts()["UNKNOWN"]
else:
    num_unknown_zip = 0    
print(f"Number of providers with known ZIP codes: {len(df_provider) - num_unknown_zip}")
print(f"Number of providers with unknown ZIP codes: {num_unknown_zip}")

Number of providers with known ZIP codes: 698
Number of providers with unknown ZIP codes: 0


Save processed data with zip codes

In [10]:
df_provider = df_provider[["THCIC_ID","PROVIDER_NAME","ZIP"]]
df_provider.to_csv(provider_output_file,index=False)